# Team 5 — Citi Bike Availability Prediction
## DATA 622: Machine Learning and Big Data

### Project Objective

This project investigates whether historical Citi Bike station data can be used to predict bike shortages approximately 30 minutes in advance.

A shortage is defined as a station having fewer than three available bikes.

### Data

- Source: Historical Citi Bike station-status records stored in Amazon S3
- Observation period: August 22–28, 2026
- Dataset size: Approximately 5.06 million observations
- Stations: Approximately 2,510
- Observation frequency: Approximately every five minutes

### Modeling Approach

The analysis includes:

1. Data quality assessment and preparation
2. Construction of a 30-minute-ahead prediction target
3. Chronological training and testing split
4. Logistic Regression and Random Forest models
5. Evaluation against a simple bike-availability rule
6. Exploration of time-of-day and station-specific shortage patterns

### Evaluation Focus

The primary objective is not simply to classify stations that already have low bike availability, but to identify stations that are likely to experience a shortage within the next 30 minutes.

Model performance is evaluated using classification metrics, emerging-shortage detection, and comparisons with a simple baseline rule.

### Limitations

The historical dataset covers approximately one week. Results are exploratory and do not yet establish production-ready predictive performance.

## 1. Citi Bike API Connection and Data Exploration

This section establishes a connection to the Citi Bike GBFS API, verifies that the API responds successfully, and explores the available data feeds.

These initial checks help us understand the data source before proceeding to historical data processing and machine learning.

In [ ]:
import requests
import pandas as pd
import matplotlib

print("All three Python packages imported successfully!")


All three Python packages imported successfully!


In [ ]:
import requests

url = "https://gbfs.citibikenyc.com/gbfs/gbfs.json"

response = requests.get(url, timeout=15)

print("HTTP status:", response.status_code)

HTTP status: 200


In [ ]:
feed = response.json()

print("Available fields:", list(feed.keys()))
print("GBFS data available:", "data" in feed)

Available fields: ['data', 'last_updated', 'ttl', 'version']
GBFS data available: True


In [ ]:
print("Available Citi Bike feeds:")

for name in feed["data"].keys():
    print(name)

Available Citi Bike feeds:
en
fr
es


In [ ]:
english_feeds = feed["data"]["en"]["feeds"]

for item in english_feeds:
    print(item["name"])

gbfs
ebikes_at_stations
system_information
station_information
station_status
free_bike_status
system_hours
system_calendar
system_regions
system_pricing_plans
system_alerts
gbfs_versions


In [ ]:
station_feed = next(
    item for item in english_feeds
    if item["name"] == "station_information"
)

station_response = requests.get(station_feed["url"], timeout=30)
station_response.raise_for_status()

stations = station_response.json()["data"]["stations"]

print("Number of Citi Bike stations:", len(stations))
print("First station:", stations[0]["name"])

Number of Citi Bike stations: 2518
First station: 3 Ave & E 109 St


In [ ]:
status_feed = next(
    item for item in english_feeds
    if item["name"] == "station_status"
)

status_response = requests.get(status_feed["url"], timeout=30)
status_response.raise_for_status()

statuses = status_response.json()["data"]["stations"]

print("Station status records:", len(statuses))
print("Available bikes at first station:", statuses[0]["num_bikes_available"])

Station status records: 2518
Available bikes at first station: 0


In [ ]:
info_df = pd.DataFrame(stations)
status_df = pd.DataFrame(statuses)

combined_df = info_df.merge(
    status_df,
    on="station_id",
    how="inner"
)

print("Stations successfully matched:", len(combined_df))

display(
    combined_df[
        ["name", "capacity", "num_bikes_available", "num_docks_available"]
    ].head(10)
)

Stations successfully matched: 2518


name,capacity,num_bikes_available,num_docks_available
3 Ave & E 109 St,25,0,0
Broadway & W 41 St,55,0,0
Prospect Pl & Nostrand Ave,0,0,0
51 Ave & 61 St,0,0,0
Vesey St & Church St,78,0,0
21 St & 20 Ave,19,0,0
Longfellow Ave & Aldus St,24,0,0
2 Ave & 36 St,43,0,0
Southern Blvd & E 149 St,0,0,0
W 12 St & Hudson St,45,0,0


## 2. Data Pipeline Setup and Validation

This section prepares the environment for working with the Team 5 Citi Bike data pipeline.

It includes file-path configuration, Python environment setup, and checks used to verify that the data collection scripts and supporting files work as expected.

In [ ]:
from pathlib import Path

project = Path("/Workspace/Users/klkid969@gmail.com/citibike-data-pipeline/teams/team-5/citibike_project")

print("Project folder exists:", project.is_dir())

for filename in [
    "download_station_info.py",
    "download_station_status.py",
    "initialize_weather_locations.py",
    "refresh_live_data.py"
]:
    print(filename, ":", (project / filename).is_file())

Project folder exists: True
download_station_info.py : True
download_station_status.py : True
initialize_weather_locations.py : True
refresh_live_data.py : True


In [ ]:
import sys

project_path = "/Workspace/Users/klkid969@gmail.com/citibike-data-pipeline/teams/team-5/citibike_project"

if project_path not in sys.path:
    sys.path.insert(0, project_path)

from download_station_info import poll_station_info

print("SUCCESS: Amish's station information module imported!")

SUCCESS: Amish's station information module imported!


In [ ]:
from download_station_status import poll_station_status

from initialize_weather_locations import (
    map_weather_cells,
    read_latest_station_information,
    write_reference_files
)

print("SUCCESS: All three project modules imported!")

SUCCESS: All three project modules imported!


In [ ]:
from download_station_info import DEFAULT_OUTPUT_ROOT
from download_station_status import DEFAULT_OUTPUT_ROOT as STATUS_ROOT
from initialize_weather_locations import DEFAULT_REFERENCE_DIRECTORY

print("Station information:", DEFAULT_OUTPUT_ROOT)
print("Station status:", STATUS_ROOT)
print("Weather reference:", DEFAULT_REFERENCE_DIRECTORY)

Station information: /Workspace/Users/klkid969@gmail.com/citibike-data-pipeline/teams/team-5/citibike_project/data/raw/station_information
Station status: /Workspace/Users/klkid969@gmail.com/citibike-data-pipeline/teams/team-5/citibike_project/data/raw/station_status
Weather reference: /Workspace/Users/klkid969@gmail.com/citibike-data-pipeline/teams/team-5/citibike_project/data/raw/weather_forecast/reference


In [ ]:
from pathlib import Path

test_folder = Path("/tmp/team5_citibike_test")

print("Temporary folder available:", test_folder.parent.is_dir())
print("Test destination:", test_folder)

Temporary folder available: True
Test destination: /tmp/team5_citibike_test


In [ ]:
from pathlib import Path

test_folder = Path("/tmp/team5_citibike_test")
test_folder.mkdir(parents=True, exist_ok=True)

test_file = test_folder / "test.txt"
test_file.write_text("Team 5 Databricks storage test")

print("File created:", test_file.exists())
print("File contents:", test_file.read_text())

File created: True
File contents: Team 5 Databricks storage test


In [ ]:
from pathlib import Path
from download_station_info import poll_station_info

output_folder = Path("/tmp/team5_citibike_test/station_information")

result = poll_station_info(
    output_folder,
    "databricks_test"
)

print("Downloaded file:", result)
print("File exists:", Path(result).exists())

Downloaded file: /tmp/team5_citibike_test/station_information/year=2026/month=11/day=06/info_20261106T030708Z.json.gz
File exists: True


In [ ]:
from pathlib import Path
from download_station_status import poll_station_status

output_folder = Path("/tmp/team5_citibike_test/station_status")

result = poll_station_status(
    output_folder,
    "databricks_test"
)

print("Downloaded file:", result)
print("File exists:", Path(result).exists())

Downloaded file: /tmp/team5_citibike_test/station_status/year=2026/month=11/day=06/status_20261106T030709Z.json.gz
File exists: True


In [ ]:
import sys
import subprocess
from pathlib import Path

project = Path("/Workspace/Users/klkid969@gmail.com/citibike-data-pipeline/teams/team-5/citibike_project")
test_root = Path("/tmp/team5_citibike_test")

command = [
    sys.executable,
    str(project / "refresh_live_data.py"),
    "--git-sha", "databricks_test",
    "--info-root", str(test_root / "station_information"),
    "--status-root", str(test_root / "station_status"),
    "--reference-directory", str(test_root / "weather_forecast/reference"),
    "--batch-size", "50",
    "--pause-seconds", "6",
]

print("Command prepared successfully.")
print("Script:", command[1])
print("Temporary output:", test_root)

Command prepared successfully.
Script: /Workspace/Users/klkid969@gmail.com/citibike-data-pipeline/teams/team-5/citibike_project/refresh_live_data.py
Temporary output: /tmp/team5_citibike_test


In [ ]:
result = subprocess.run(
    command,
    capture_output=True,
    text=True,
    timeout=900
)

print("Exit code:", result.returncode)
print("\nPIPELINE OUTPUT:")
print(result.stdout)

if result.stderr:
    print("\nERRORS OR WARNINGS:")
    print(result.stderr)

Exit code: 0

PIPELINE OUTPUT:
Generated-data directories are ready.
Saved: /tmp/team5_citibike_test/station_information/year=2026/month=11/day=06/info_20261106T030711Z.json.gz
Saved: /tmp/team5_citibike_test/station_status/year=2026/month=11/day=06/status_20261106T030711Z.json.gz
Rebuilding weather mapping: new or changed station coordinates.
Mapped 50/2,518
Mapped 100/2,518
Mapped 150/2,518
Mapped 200/2,518
Mapped 250/2,518
Mapped 300/2,518
Mapped 350/2,518
Mapped 400/2,518
Mapped 450/2,518
Mapped 500/2,518
Mapped 550/2,518
Mapped 600/2,518
Mapped 650/2,518
Mapped 700/2,518
Mapped 750/2,518
Mapped 800/2,518
Mapped 850/2,518
Mapped 900/2,518
Mapped 950/2,518
Mapped 1,000/2,518
Mapped 1,050/2,518
Mapped 1,100/2,518
Mapped 1,150/2,518
Mapped 1,200/2,518
Mapped 1,250/2,518
Mapped 1,300/2,518
Mapped 1,350/2,518
Mapped 1,400/2,518
Mapped 1,450/2,518
Mapped 1,500/2,518
Mapped 1,550/2,518
Mapped 1,600/2,518
Mapped 1,650/2,518
Mapped 1,700/2,518
Mapped 1,750/2,518
Mapped 1,800/2,518
Mapped 1,

In [ ]:
from pathlib import Path

root = Path("/tmp/team5_citibike_test")

files = list(root.rglob("*.json.gz"))
files += list(root.rglob("*.json"))
files += list(root.rglob("*.csv"))

print("Total generated files:", len(files))

for file in files:
    print(file.relative_to(root), "-", file.stat().st_size, "bytes")

Total generated files: 6
station_status/year=2026/month=11/day=06/status_20261106T030711Z.json.gz - 73296 bytes
station_status/year=2026/month=11/day=06/status_20261106T030709Z.json.gz - 73296 bytes
station_information/year=2026/month=11/day=06/info_20261106T030708Z.json.gz - 132366 bytes
station_information/year=2026/month=11/day=06/info_20261106T030711Z.json.gz - 132365 bytes
weather_forecast/reference/hrrr_locations.json - 6661 bytes
weather_forecast/reference/station_weather_lookup.csv - 205053 bytes


In [ ]:
from pathlib import Path

rainfall_file = Path(
    "/Workspace/Users/klkid969@gmail.com/Team5_Masoud_Data_Test/rainfall.csv"
)

print("Rainfall file exists:", rainfall_file.is_file())

Rainfall file exists: True


## 3. Weather and Station Data Exploration

This section examines supporting datasets used in the Team 5 Citi Bike project, including rainfall measurements, weather-grid identifiers, and station information.

The analysis checks dataset structure, record counts, and station coverage to help determine how these data sources could support future modeling and pipeline integration.

In [ ]:
import pandas as pd

rainfall_df = pd.read_csv(rainfall_file)

print("Rows and columns:", rainfall_df.shape)
print("\nColumn names:")
print(rainfall_df.columns.tolist())

display(rainfall_df.head(5))

/home/spark-45b83057-1b47-4e05-bef7-7a/.ipykernel/63/command-8182566242976298-3938668292:3: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,2

Rows and columns: (52978, 1)

Column names:
['rainfall']


rainfall
g40.915N73.925W
0.000000
0.000000
0.000000
0.000000


In [ ]:
rainfall_df = pd.read_csv(rainfall_file, skiprows=1)

print("Rows and columns:", rainfall_df.shape)
print("Column names:", rainfall_df.columns.tolist())

display(rainfall_df.head(5))

Rows and columns: (52977, 641)
Column names: ['datetime_utc', 'g40.595N73.995W', 'g40.595N74.005W', 'g40.605N73.965W', 'g40.605N73.975W', 'g40.605N73.985W', 'g40.605N73.995W', 'g40.605N74.005W', 'g40.605N74.015W', 'g40.605N74.025W', 'g40.605N74.035W', 'g40.615N73.915W', 'g40.615N73.925W', 'g40.615N73.935W', 'g40.615N73.945W', 'g40.615N73.955W', 'g40.615N73.965W', 'g40.615N73.975W', 'g40.615N73.985W', 'g40.615N73.995W', 'g40.615N74.005W', 'g40.615N74.015W', 'g40.615N74.025W', 'g40.615N74.035W', 'g40.615N74.045W', 'g40.625N73.885W', 'g40.625N73.895W', 'g40.625N73.905W', 'g40.625N73.915W', 'g40.625N73.925W', 'g40.625N73.935W', 'g40.625N73.945W', 'g40.625N73.955W', 'g40.625N73.965W', 'g40.625N73.975W', 'g40.625N73.985W', 'g40.625N73.995W', 'g40.625N74.005W', 'g40.625N74.015W', 'g40.625N74.025W', 'g40.625N74.035W', 'g40.625N74.045W', 'g40.635N73.855W', 'g40.635N73.865W', 'g40.635N73.875W', 'g40.635N73.885W', 'g40.635N73.895W', 'g40.635N73.905W', 'g40.635N73.915W', 'g40.635N73.925W', 'g40.63

datetime_utc,g40.595N73.995W,g40.595N74.005W,g40.605N73.965W,g40.605N73.975W,g40.605N73.985W,g40.605N73.995W,g40.605N74.005W,g40.605N74.015W,g40.605N74.025W,g40.605N74.035W,g40.615N73.915W,g40.615N73.925W,g40.615N73.935W,g40.615N73.945W,g40.615N73.955W,g40.615N73.965W,g40.615N73.975W,g40.615N73.985W,g40.615N73.995W,g40.615N74.005W,g40.615N74.015W,g40.615N74.025W,g40.615N74.035W,g40.615N74.045W,g40.625N73.885W,g40.625N73.895W,g40.625N73.905W,g40.625N73.915W,g40.625N73.925W,g40.625N73.935W,g40.625N73.945W,g40.625N73.955W,g40.625N73.965W,g40.625N73.975W,g40.625N73.985W,g40.625N73.995W,g40.625N74.005W,g40.625N74.015W,g40.625N74.025W,g40.625N74.035W,g40.625N74.045W,g40.635N73.855W,g40.635N73.865W,g40.635N73.875W,g40.635N73.885W,g40.635N73.895W,g40.635N73.905W,g40.635N73.915W,g40.635N73.925W,g40.635N73.935W,g40.635N73.945W,g40.635N73.955W,g40.635N73.965W,g40.635N73.975W,g40.635N73.985W,g40.635N73.995W,g40.635N74.005W,g40.635N74.015W,g40.635N74.025W,g40.635N74.035W,g40.635N74.045W,g40.645N73.845W,g40.645N73.855W,g40.645N73.865W,g40.645N73.875W,g40.645N73.885W,g40.645N73.895W,g40.645N73.905W,g40.645N73.915W,g40.645N73.925W,g40.645N73.935W,g40.645N73.945W,g40.645N73.955W,g40.645N73.965W,g40.645N73.975W,g40.645N73.985W,g40.645N73.995W,g40.645N74.005W,g40.645N74.015W,g40.645N74.025W,g40.645N74.035W,g40.655N73.845W,g40.655N73.855W,g40.655N73.865W,g40.655N73.875W,g40.655N73.885W,g40.655N73.895W,g40.655N73.905W,g40.655N73.915W,g40.655N73.925W,g40.655N73.935W,g40.655N73.945W,g40.655N73.955W,g40.655N73.965W,g40.655N73.975W,g40.655N73.985W,g40.655N73.995W,g40.655N74.005W,g40.655N74.015W,g40.655N74.025W,g40.665N73.845W,g40.665N73.855W,g40.665N73.865W,g40.665N73.875W,g40.665N73.885W,g40.665N73.895W,g40.665N73.905W,g40.665N73.915W,g40.665N73.925W,g40.665N73.935W,g40.665N73.945W,g40.665N73.955W,g40.665N73.965W,g40.665N73.975W,g40.665N73.985W,g40.665N73.995W,g40.665N74.005W,g40.665N74.015W,g40.665N74.065W,g40.665N74.075W,g40.665N74.085W,g40.665N74.095W,g40.665N74.105W,g40.665N74.115W,g40.675N73.845W,g40.675N73.855W,g40.675N73.865W,g40.675N73.875W,g40.675N73.885W,g40.675N73.895W,g40.675N73.905W,g40.675N73.915W,g40.675N73.925W,g40.675N73.935W,g40.675N73.945W,g40.675N73.955W,g40.675N73.965W,g40.675N73.975W,g40.675N73.985W,g40.675N73.995W,g40.675N74.005W,g40.675N74.015W,g40.675N74.065W,g40.675N74.075W,g40.675N74.085W,g40.675N74.095W,g40.675N74.105W,g40.675N74.115W,g40.675N74.125W,g40.685N73.815W,g40.685N73.825W,g40.685N73.835W,g40.685N73.845W,g40.685N73.855W,g40.685N73.865W,g40.685N73.875W,g40.685N73.885W,g40.685N73.895W,g40.685N73.905W,g40.685N73.915W,g40.685N73.925W,g40.685N73.935W,g40.685N73.945W,g40.685N73.955W,g40.685N73.965W,g40.685N73.975W,g40.685N73.985W,g40.685N73.995W,g40.685N74.005W,g40.685N74.015W,g40.685N74.025W,g40.685N74.045W,g40.685N74.065W,g40.685N74.075W,g40.685N74.085W,g40.685N74.095W,g40.685N74.105W,g40.685N74.115W,g40.695N73.795W,g40.695N73.805W,g40.695N73.815W,g40.695N73.825W,g40.695N73.835W,g40.695N73.845W,g40.695N73.855W,g40.695N73.865W,g40.695N73.875W,g40.695N73.885W,g40.695N73.895W,g40.695N73.905W,g40.695N73.915W,g40.695N73.925W,g40.695N73.935W,g40.695N73.945W,g40.695N73.955W,g40.695N73.965W,g40.695N73.975W,g40.695N73.985W,g40.695N73.995W,g40.695N74.005W,g40.695N74.015W,g40.695N74.025W,g40.695N74.035W,g40.695N74.045W,g40.695N74.055W,g40.695N74.065W,g40.695N74.075W,g40.695N74.085W,g40.695N74.095W,g40.695N74.105W,g40.695N74.115W,g40.705N73.795W,g40.705N73.805W,g40.705N73.815W,g40.705N73.825W,g40.705N73.835W,g40.705N73.845W,g40.705N73.855W,g40.705N73.865W,g40.705N73.875W,g40.705N73.885W,g40.705N73.895W,g40.705N73.905W,g40.705N73.915W,g40.705N73.925W,g40.705N73.935W,g40.705N73.945W,g40.705N73.955W,g40.705N73.965W,g40.705N73.975W,g40.705N73.985W,g40.705N73.995W,g40.705N74.005W,g40.705N74.015W,g40.705N74.035W,g40.705N74.045W,g40.705N74.055W,g40.705N74.065W,g40.705N74.075W,g40.705N74.085W,g40.705N74.095W,g40.705N74.105W,g40.715N73.795W,g40.715N73.805W,g40.715N73.815W,g40.715N73.825W,g40.715N73.835W,g40.715N73.845W,g40.715N73.855W,g40

In [ ]:
print("Rows:", len(rainfall_df))
print("Columns:", len(rainfall_df.columns))
print("First date:", rainfall_df["datetime_utc"].iloc[0])
print("Last date:", rainfall_df["datetime_utc"].iloc[-1])

Rows: 52977
Columns: 641
First date: 2020-10-14 00:00
Last date: 2026-11-02 23:00


In [ ]:
from pathlib import Path
import pandas as pd

lookup_file = Path(
    "/Workspace/Users/klkid969@gmail.com/Team5_Masoud_Data_Test/station_grid_lookup.csv"
)

lookup_df = pd.read_csv(lookup_file, skiprows=1)

print("Rows and columns:", lookup_df.shape)
print("Column names:", lookup_df.columns.tolist())

display(lookup_df.head(5))

Rows and columns: (2526, 5)
Column names: ['station_id', 'lat', 'lon', 'grid_id', 'distance_km']


station_id,lat,lon,grid_id,distance_km
1835246516012366338,40.7564,-73.87504,g40.755N73.875W,0.156
66dd1585-0aca-11e7-82f6-3863bb44ef7c,40.71924,-73.95242,g40.715N73.955W,0.52
66dd4f31-0aca-11e7-82f6-3863bb44ef7c,40.78499979,-73.97283406,g40.785N73.975W,0.183
2195265983529389682,40.70223,-74.08899,g40.705N74.085W,0.457
2170352212111402482,40.714063,-74.014432,g40.715N74.015W,0.115


In [ ]:
station_grids = set(lookup_df["grid_id"].dropna().astype(str))
rainfall_grids = set(rainfall_df.columns) - {"datetime_utc"}

matching_grids = station_grids & rainfall_grids

print("Unique station grid IDs:", len(station_grids))
print("Rainfall grid columns:", len(rainfall_grids))
print("Matching grid IDs:", len(matching_grids))
print("Station grids missing from rainfall:", len(station_grids - rainfall_grids))

Unique station grid IDs: 327
Rainfall grid columns: 640
Matching grid IDs: 327
Station grids missing from rainfall: 0


In [ ]:
import pandas as pd
from pathlib import Path

station_file = Path(
    "/Workspace/Users/klkid969@gmail.com/Team5_Masoud_Data_Test/station_info.parquet"
)

station_df = pd.read_parquet(station_file)

print("Rows and columns:", station_df.shape)
print("Column names:", station_df.columns.tolist())

display(station_df.head(5))

Rows and columns: (138282, 26)
Column names: ['fetched_at_raw', 'fetched_at', 'feed_ts', 'poller_version', 'git_sha', 'capacity', 'eightd_has_key_dispenser', 'eightd_station_services', 'electric_bike_surcharge_waiver', 'external_id', 'has_kiosk', 'lat', 'lon', 'name', 'region_id', 'rental_methods', 'rental_uris', 'short_name', 'station_id', 'station_type', 'snapshot_date', '_has_parse_error', 'feed_updated_at', '_required_value_missing', '_has_negative_capacity', '_has_invalid_coordinates']


fetched_at_raw,fetched_at,feed_ts,poller_version,git_sha,capacity,eightd_has_key_dispenser,eightd_station_services,electric_bike_surcharge_waiver,external_id,has_kiosk,lat,lon,name,region_id,rental_methods,rental_uris,short_name,station_id,station_type,snapshot_date,_has_parse_error,feed_updated_at,_required_value_missing,_has_negative_capacity,_has_invalid_coordinates
2026-10-11T01:43:24.618338+00:00,2026-10-11T01:43:24.618Z,1791682972,1.2,manual_run,0,false,[],false,1822663031356509142,true,40.64235,-73.96951,Matthews Ct & Coney Island Ave,71,"[""KEY"",""CREDITCARD""]","{""android"":""https://bkn.lft.to/lastmile_qr_scan"",""ios"":""https://bkn.lft.to/lastmile_qr_scan""}",3034.02,1822663031356509142,classic,2026-10-11,false,2026-10-11T01:42:52.000Z,false,false,false
2026-10-11T01:43:24.618338+00:00,2026-10-11T01:43:24.618Z,1791682972,1.2,manual_run,19,false,[],false,7a6fc75f-a605-4dd6-b567-b2bbbfb862f3,true,40.73145,-73.90022,65 Pl & 53 Ave,71,"[""KEY"",""CREDITCARD""]","{""android"":""https://bkn.lft.to/lastmile_qr_scan"",""ios"":""https://bkn.lft.to/lastmile_qr_scan""}",5764.02,7a6fc75f-a605-4dd6-b567-b2bbbfb862f3,classic,2026-10-11,false,2026-10-11T01:42:52.000Z,false,false,false
2026-10-11T01:43:24.618338+00:00,2026-10-11T01:43:24.618Z,1791682972,1.2,manual_run,0,false,[],false,2235294222661977076,false,40.66363,-73.90231,Livonia Ave & Powell St,71,"[""KEY"",""CREDITCARD""]","{""android"":""https://bkn.lft.to/lastmile_qr_scan"",""ios"":""https://bkn.lft.to/lastmile_qr_scan""}",3678.05,2235294222661977076,classic,2026-10-11,false,2026-10-11T01:42:52.000Z,false,false,false
2026-10-11T01:43:24.618338+00:00,2026-10-11T01:43:24.618Z,1791682972,1.2,manual_run,78,false,[],false,06439006-11b6-44f0-8545-c9d39035f32a,true,40.71221958009051,-74.01047229766846,Vesey St & Church St,71,"[""KEY"",""CREDITCARD""]","{""android"":""https://bkn.lft.to/lastmile_qr_scan"",""ios"":""https://bkn.lft.to/lastmile_qr_scan""}",5216.06,06439006-11b6-44f0-8545-c9d39035f32a,classic,2026-10-11,false,2026-10-11T01:42:52.000Z,false,false,false
2026-10-11T01:43:24.618338+00:00,2026-10-11T01:43:24.618Z,1791682972,1.2,manual_run,19,false,[],false,1986707977001495180,true,40.78393602019729,-73.91273114065842,21 St & 20 Ave,71,"[""KEY"",""CREDITCARD""]","{""android"":""https://bkn.lft.to/lastmile_qr_scan"",""ios"":""https://bkn.lft.to/lastmile_qr_scan""}",7363.03,1986707977001495180,classic,2026-10-11,false,2026-10-11T01:42:52.000Z,false,false,false


In [ ]:
print("Unique stations:", station_df["station_id"].nunique())
print("First snapshot:", station_df["fetched_at"].min())
print("Last snapshot:", station_df["fetched_at"].max())
print("Unique snapshot dates:", station_df["snapshot_date"].nunique())


Unique stations: 2526
First snapshot: 2026-08-21 22:42:03.322032
Last snapshot: 2026-10-12 01:43:24.142498
Unique snapshot dates: 53


In [ ]:
print("Station information columns containing 'bike':")
print([col for col in station_df.columns if "bike" in col.lower()])

print("\nStation information columns containing 'available':")
print([col for col in station_df.columns if "available" in col.lower()])

Station information columns containing 'bike':
['electric_bike_surcharge_waiver']

Station information columns containing 'available':
[]


## 4. Historical Citi Bike Data Loading and Validation

This section loads historical Citi Bike station-status records from Amazon S3 into Apache Spark.

The analysis examines the dataset's structure, observation period, station coverage, and data quality before preparing the records for machine learning.

Data validation includes checking for missing values, duplicate observations, and invalid bike availability counts.

In [ ]:
s3_path = "s3://cuny-sps-citibike-archive/exports/station_status/"

try:
    files = dbutils.fs.ls(s3_path)
    print("SUCCESS! S3 folder is accessible.")
    print("Items found:", len(files))

    for file in files[:10]:
        print(file.name)

except Exception as e:
    print("Unable to access S3 folder.")
    print("Error:", str(e)[:500])

SUCCESS! S3 folder is accessible.
Items found: 74
_SUCCESS
nyc_date=2026-08-21/
nyc_date=2026-08-22/
nyc_date=2026-08-23/
nyc_date=2026-08-24/
nyc_date=2026-08-25/
nyc_date=2026-08-26/
nyc_date=2026-08-27/
nyc_date=2026-08-28/
nyc_date=2026-08-29/


In [ ]:
day_path = s3_path + "nyc_date=2026-08-21/"

day_files = dbutils.fs.ls(day_path)

print("Files found:", len(day_files))

for file in day_files[:10]:
    print(file.name, "-", file.size, "bytes")

Files found: 4
_SUCCESS - 0 bytes
_committed_6904995139006545167 - 124 bytes
_started_6904995139006545167 - 0 bytes
part-00029-tid-6904995139006545167-cc481743-7908-4d80-ba0a-6d198b24f0e2-186-2.c000.snappy.parquet - 1705698 bytes


In [ ]:
status_day_df = spark.read.parquet(day_path)

print("Number of records:", status_day_df.count())
print("Column names:", status_day_df.columns)

display(status_day_df.limit(5))

Number of records: 158004
Column names: ['fetched_at_raw', 'fetched_at', 'poller_version', 'git_sha', 'feed_ts', 'eightd_has_available_keys', 'is_installed', 'is_renting', 'is_returning', 'last_reported', 'legacy_id', 'num_bikes_available', 'num_bikes_disabled', 'num_docks_available', 'num_docks_disabled', 'num_ebikes_available', 'num_scooters_available', 'num_scooters_unavailable', 'station_id', '_has_parse_error', 'feed_updated_at', 'station_reported_at', '_last_reported_placeholder', '_required_value_missing', '_has_negative_count', '_ebikes_exceed_total']


fetched_at_raw,fetched_at,poller_version,git_sha,feed_ts,eightd_has_available_keys,is_installed,is_renting,is_returning,last_reported,legacy_id,num_bikes_available,num_bikes_disabled,num_docks_available,num_docks_disabled,num_ebikes_available,num_scooters_available,num_scooters_unavailable,station_id,_has_parse_error,feed_updated_at,station_reported_at,_last_reported_placeholder,_required_value_missing,_has_negative_count,_ebikes_exceed_total
2026-08-21T23:16:47.872416+00:00,2026-08-21T23:16:47.872Z,null,null,1787354157,false,true,true,true,1787354139,2108372356498784942,7,3,4,0,3,0,0,2108372356498784942,false,2026-08-21T23:15:57.000Z,2026-08-21T23:15:39.000Z,false,false,false,false
2026-08-21T23:16:47.872416+00:00,2026-08-21T23:16:47.872Z,null,null,1787354157,false,true,true,true,1787354137,4536,12,3,9,0,4,0,0,cbc48689-7805-49dd-9669-5471f4b7b6fb,false,2026-08-21T23:15:57.000Z,2026-08-21T23:15:37.000Z,false,false,false,false
2026-08-21T23:16:47.872416+00:00,2026-08-21T23:16:47.872Z,null,null,1787354157,false,true,true,true,1787354135,4649,13,1,14,0,0,0,0,bd6f422b-d7ae-4d7e-9261-653fdd8e6888,false,2026-08-21T23:15:57.000Z,2026-08-21T23:15:35.000Z,false,false,false,false
2026-08-21T23:16:47.872416+00:00,2026-08-21T23:16:47.872Z,null,null,1787354157,false,true,true,true,1787354132,3270,1,1,12,0,0,0,0,66ddd93e-0aca-11e7-82f6-3863bb44ef7c,false,2026-08-21T23:15:57.000Z,2026-08-21T23:15:32.000Z,false,false,false,false
2026-08-21T23:16:47.872416+00:00,2026-08-21T23:16:47.872Z,null,null,1787354157,false,true,true,true,1787354120,3269,1,2,10,0,1,0,0,66ddd8a8-0aca-11e7-82f6-3863bb44ef7c,false,2026-08-21T23:15:57.000Z,2026-08-21T23:15:20.000Z,false,false,false,false


In [ ]:
from pyspark.sql import functions as F

status_day_df.select(
    F.countDistinct("station_id").alias("unique_stations"),
    F.min("fetched_at").alias("first_record"),
    F.max("fetched_at").alias("last_record")
).show(truncate=False)

+---------------+-------------------------+--------------------------+
|unique_stations|first_record             |last_record               |
+---------------+-------------------------+--------------------------+
|2508           |2026-08-21 21:40:37.96825|2026-08-22 03:56:47.672711|
+---------------+-------------------------+--------------------------+



In [ ]:
status_day_df.select(
    "station_id",
    "fetched_at",
    "num_bikes_available"
).filter(
    F.col("station_id").isNotNull()
).orderBy(
    "station_id", "fetched_at"
).show(10, truncate=False)

+------------------------------------+--------------------------+-------------------+
|station_id                          |fetched_at                |num_bikes_available|
+------------------------------------+--------------------------+-------------------+
|00284700-9d22-42ce-8485-113fed9879c1|2026-08-21 21:40:37.96825 |1                  |
|00284700-9d22-42ce-8485-113fed9879c1|2026-08-21 22:41:47.283673|3                  |
|00284700-9d22-42ce-8485-113fed9879c1|2026-08-21 22:46:46.414785|3                  |
|00284700-9d22-42ce-8485-113fed9879c1|2026-08-21 22:51:46.238585|2                  |
|00284700-9d22-42ce-8485-113fed9879c1|2026-08-21 22:56:45.589474|2                  |
|00284700-9d22-42ce-8485-113fed9879c1|2026-08-21 23:11:56.894243|1                  |
|00284700-9d22-42ce-8485-113fed9879c1|2026-08-21 23:16:47.872416|1                  |
|00284700-9d22-42ce-8485-113fed9879c1|2026-08-21 23:21:47.123477|1                  |
|00284700-9d22-42ce-8485-113fed9879c1|2026-08-21 23:26

In [ ]:
date_folders = [
    f.name.rstrip("/")
    for f in files
    if f.name.startswith("nyc_date=")
]

print("Number of date folders:", len(date_folders))
print("Earliest date:", min(date_folders))
print("Latest date:", max(date_folders))


Number of date folders: 73
Earliest date: nyc_date=2026-08-21
Latest date: nyc_date=2026-11-01


In [ ]:
status_day_df.select(
    F.count("*").alias("total_records"),
    F.sum(F.col("num_bikes_available").isNull().cast("int")).alias("missing_bike_counts"),
    F.sum(F.col("station_id").isNull().cast("int")).alias("missing_station_ids"),
    F.sum(F.col("fetched_at").isNull().cast("int")).alias("missing_timestamps")
).show()

+-------------+-------------------+-------------------+------------------+
|total_records|missing_bike_counts|missing_station_ids|missing_timestamps|
+-------------+-------------------+-------------------+------------------+
|       158004|                  0|                  0|                 0|
+-------------+-------------------+-------------------+------------------+



In [ ]:
duplicates = (
    status_day_df
    .groupBy("station_id", "fetched_at")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate station/timestamp combinations:", duplicates.count())

Duplicate station/timestamp combinations: 0


In [ ]:
status_day_df.select(
    F.sum(
        (F.col("num_bikes_available") < 0).cast("int")
    ).alias("negative_bike_counts"),
    F.sum(
        F.col("_has_parse_error").cast("int")
    ).alias("parse_errors"),
    F.sum(
        F.col("_has_negative_count").cast("int")
    ).alias("flagged_negative_counts")
).show()

+--------------------+------------+-----------------------+
|negative_bike_counts|parse_errors|flagged_negative_counts|
+--------------------+------------+-----------------------+
|                   0|           0|                      0|
+--------------------+------------+-----------------------+



In [ ]:
station_counts = (
    status_day_df
    .groupBy("station_id")
    .count()
)

station_counts.select(
    F.min("count").alias("fewest_observations"),
    F.max("count").alias("most_observations"),
    F.round(F.avg("count"), 1).alias("average_observations")
).show()

+-------------------+-----------------+--------------------+
|fewest_observations|most_observations|average_observations|
+-------------------+-----------------+--------------------+
|                 63|               63|                63.0|
+-------------------+-----------------+--------------------+



In [ ]:
from pyspark.sql.window import Window

window = Window.partitionBy("station_id").orderBy("fetched_at")

intervals = (
    status_day_df
    .withColumn("previous_time", F.lag("fetched_at").over(window))
    .withColumn(
        "minutes_between",
        (F.unix_timestamp("fetched_at") -
         F.unix_timestamp("previous_time")) / 60
    )
)

intervals.select(
    F.round(F.min("minutes_between"), 2).alias("shortest_interval"),
    F.round(F.avg("minutes_between"), 2).alias("average_interval"),
    F.round(F.max("minutes_between"), 2).alias("longest_interval")
).show()

+-----------------+----------------+----------------+
|shortest_interval|average_interval|longest_interval|
+-----------------+----------------+----------------+
|             4.85|            6.07|           61.17|
+-----------------+----------------+----------------+



In [ ]:
valid_intervals = intervals.filter(
    F.col("minutes_between").isNotNull()
)

valid_intervals.select(
    F.count("*").alias("total_intervals"),
    F.sum(
        (F.col("minutes_between") > 10).cast("int")
    ).alias("gaps_over_10_minutes"),
    F.round(
        F.avg(
            (F.col("minutes_between") > 10).cast("int")
        ) * 100, 2
    ).alias("percentage_with_gaps")
).show()

+---------------+--------------------+--------------------+
|total_intervals|gaps_over_10_minutes|percentage_with_gaps|
+---------------+--------------------+--------------------+
|         155496|                5016|                3.23|
+---------------+--------------------+--------------------+



In [ ]:
long_gaps = intervals.filter(
    F.col("minutes_between") > 10
)

long_gaps.groupBy(
    "previous_time", "fetched_at"
).agg(
    F.countDistinct("station_id").alias("stations_affected")
).orderBy(
    F.desc("stations_affected")
).show(10, truncate=False)

+--------------------------+--------------------------+-----------------+
|previous_time             |fetched_at                |stations_affected|
+--------------------------+--------------------------+-----------------+
|2026-08-21 21:40:37.96825 |2026-08-21 22:41:47.283673|2508             |
|2026-08-21 22:56:45.589474|2026-08-21 23:11:56.894243|2508             |
+--------------------------+--------------------------+-----------------+



In [ ]:
next_day_path = s3_path + "nyc_date=2026-08-22/"

next_day_df = spark.read.parquet(next_day_path)

print("Total records:", next_day_df.count())
print(
    "Unique stations:",
    next_day_df.select("station_id").distinct().count()
)

next_day_df.select(
    F.min("fetched_at").alias("first_record"),
    F.max("fetched_at").alias("last_record")
).show(truncate=False)

Total records: 722304
Unique stations: 2508
+--------------------------+--------------------------+
|first_record              |last_record               |
+--------------------------+--------------------------+
|2026-08-22 04:01:47.873904|2026-08-23 03:56:46.598301|
+--------------------------+--------------------------+



In [ ]:
next_window = Window.partitionBy("station_id").orderBy("fetched_at")

next_intervals = (
    next_day_df
    .withColumn("previous_time", F.lag("fetched_at").over(next_window))
    .withColumn(
        "minutes_between",
        (F.unix_timestamp("fetched_at") -
         F.unix_timestamp("previous_time")) / 60
    )
)

next_intervals.select(
    F.round(F.min("minutes_between"), 2).alias("shortest"),
    F.round(F.avg("minutes_between"), 2).alias("average"),
    F.round(F.max("minutes_between"), 2).alias("longest")
).show()

+--------+-------+-------+
|shortest|average|longest|
+--------+-------+-------+
|    4.52|    5.0|   5.48|
+--------+-------+-------+



In [ ]:
next_day_df.select(
    F.count("*").alias("total_records"),
    F.sum(F.col("station_id").isNull().cast("int")).alias("missing_station_ids"),
    F.sum(F.col("fetched_at").isNull().cast("int")).alias("missing_timestamps"),
    F.sum(F.col("num_bikes_available").isNull().cast("int")).alias("missing_bike_counts"),
    F.sum(F.col("_has_parse_error").cast("int")).alias("parse_errors")
).show()

+-------------+-------------------+------------------+-------------------+------------+
|total_records|missing_station_ids|missing_timestamps|missing_bike_counts|parse_errors|
+-------------+-------------------+------------------+-------------------+------------+
|       722304|                  0|                 0|                  0|           0|
+-------------+-------------------+------------------+-------------------+------------+



In [ ]:
next_day_df.select(
    F.count("*").alias("total_observations"),
    F.sum(
        (F.col("num_bikes_available") < 3).cast("int")
    ).alias("low_bike_observations"),
    F.round(
        F.avg(
            (F.col("num_bikes_available") < 3).cast("int")
        ) * 100, 2
    ).alias("low_bike_percentage")
).show()

+------------------+---------------------+-------------------+
|total_observations|low_bike_observations|low_bike_percentage|
+------------------+---------------------+-------------------+
|            722304|                99458|              13.77|
+------------------+---------------------+-------------------+



In [ ]:
next_day_df.select(
    F.countDistinct("station_id").alias("total_stations"),
    F.countDistinct(
        F.when(
            F.col("num_bikes_available") < 3,
            F.col("station_id")
        )
    ).alias("stations_with_low_availability")
).show()

+--------------+------------------------------+
|total_stations|stations_with_low_availability|
+--------------+------------------------------+
|          2508|                          1031|
+--------------+------------------------------+



## 5. Prediction Target and Machine Learning Preparation

This section prepares historical Citi Bike observations for predicting station shortages approximately 30 minutes into the future.

A shortage is defined as fewer than three available bikes. Using time-ordered station observations, the analysis constructs the prediction target and prepares the data for machine learning.

The data is divided chronologically into training and testing periods to evaluate predictions on later observations and reduce the risk of future-data leakage.

In [ ]:
from pyspark.sql.window import Window

prediction_window = (
    Window.partitionBy("station_id")
    .orderBy("fetched_at")
)

model_df = (
    next_day_df
    .withColumn(
        "bikes_30min_ahead",
        F.lead("num_bikes_available", 6).over(prediction_window)
    )
    .withColumn(
        "time_30min_ahead",
        F.lead("fetched_at", 6).over(prediction_window)
    )
    .withColumn(
        "low_bikes_30min",
        F.when(F.col("bikes_30min_ahead") < 3, 1)
         .when(F.col("bikes_30min_ahead").isNotNull(), 0)
    )
)

model_df.select(
    "station_id",
    "num_bikes_available",
    "bikes_30min_ahead",
    "low_bikes_30min"
).show(10, truncate=False)

+------------------------------------+-------------------+-----------------+---------------+
|station_id                          |num_bikes_available|bikes_30min_ahead|low_bikes_30min|
+------------------------------------+-------------------+-----------------+---------------+
|0096fecb-3a56-4585-9204-aa9554f7bc49|37                 |25               |0              |
|0096fecb-3a56-4585-9204-aa9554f7bc49|37                 |25               |0              |
|0096fecb-3a56-4585-9204-aa9554f7bc49|30                 |24               |0              |
|0096fecb-3a56-4585-9204-aa9554f7bc49|30                 |24               |0              |
|0096fecb-3a56-4585-9204-aa9554f7bc49|27                 |21               |0              |
|0096fecb-3a56-4585-9204-aa9554f7bc49|25                 |18               |0              |
|0096fecb-3a56-4585-9204-aa9554f7bc49|25                 |18               |0              |
|0096fecb-3a56-4585-9204-aa9554f7bc49|25                 |17          

In [ ]:
model_df.groupBy("low_bikes_30min").count().orderBy(
    "low_bikes_30min"
).show()

+---------------+------+
|low_bikes_30min| count|
+---------------+------+
|           NULL| 15048|
|              0|609722|
|              1| 97534|
+---------------+------+



In [ ]:
model_df.withColumn(
    "minutes_ahead",
    (
        F.unix_timestamp("time_30min_ahead")
        - F.unix_timestamp("fetched_at")
    ) / 60
).select(
    F.round(F.min("minutes_ahead"), 2).alias("shortest"),
    F.round(F.avg("minutes_ahead"), 2).alias("average"),
    F.round(F.max("minutes_ahead"), 2).alias("longest")
).show()

+--------+-------+-------+
|shortest|average|longest|
+--------+-------+-------+
|   29.55|   30.0|  30.52|
+--------+-------+-------+



In [ ]:
training_df = model_df.filter(
    F.col("low_bikes_30min").isNotNull()
)

print("Training examples:", training_df.count())

training_df.groupBy("low_bikes_30min").count().show()

Training examples: 707256
+---------------+------+
|low_bikes_30min| count|
+---------------+------+
|              1| 97534|
|              0|609722|
+---------------+------+



In [ ]:
training_df.select(
    F.min("fetched_at").alias("earliest_observation"),
    F.max("fetched_at").alias("latest_observation"),
    F.countDistinct("fetched_at").alias("unique_collection_times")
).show(truncate=False)

+--------------------------+-------------------------+-----------------------+
|earliest_observation      |latest_observation       |unique_collection_times|
+--------------------------+-------------------------+-----------------------+
|2026-08-22 04:01:47.873904|2026-08-23 03:26:46.90815|282                    |
+--------------------------+-------------------------+-----------------------+



In [ ]:
from pyspark.sql import functions as F

times = (
    training_df
    .select("fetched_at")
    .distinct()
    .orderBy("fetched_at")
    .collect()
)

split_index = int(len(times) * 0.70)
split_time = times[split_index]["fetched_at"]

train_df = training_df.filter(
    F.col("time_30min_ahead") < F.lit(split_time)
)

test_df = training_df.filter(
    F.col("fetched_at") >= F.lit(split_time)
)

print("Split time:", split_time)
print("Training records:", train_df.count())
print("Testing records:", test_df.count())

Split time: 2026-08-22 20:26:44.453023
Training records: 479028
Testing records: 213180


In [ ]:
print("TRAINING DATA")
train_df.groupBy("low_bikes_30min").count().show()

print("TESTING DATA")
test_df.groupBy("low_bikes_30min").count().show()

TRAINING DATA
+---------------+------+
|low_bikes_30min| count|
+---------------+------+
|              1| 62961|
|              0|416067|
+---------------+------+

TESTING DATA
+---------------+------+
|low_bikes_30min| count|
+---------------+------+
|              1| 32042|
|              0|181138|
+---------------+------+



In [ ]:
from pyspark.sql import functions as F

baseline_results = test_df.withColumn(
    "baseline_prediction",
    F.lit(0)
)

baseline_results.select(
    F.round(
        F.avg(
            (F.col("baseline_prediction") == F.col("low_bikes_30min"))
            .cast("int")
        ) * 100, 2
    ).alias("baseline_accuracy_percent")
).show()

+-------------------------+
|baseline_accuracy_percent|
+-------------------------+
|                    84.97|
+-------------------------+



## 6. Machine Learning Model Development

This section develops and evaluates machine learning models for predicting Citi Bike station shortages.

Logistic Regression serves as our initial classification model, followed by experiments incorporating recent bike availability trends and time-of-day features.

A Random Forest classifier is also evaluated to investigate whether a nonlinear model can improve predictions.

The models are compared with a simple bike-count warning rule to determine whether machine learning provides additional value beyond current station availability.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

assembler = VectorAssembler(
    inputCols=["num_bikes_available"],
    outputCol="features"
)

train_ready = assembler.transform(train_df)
test_ready = assembler.transform(test_df)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="low_bikes_30min",
    maxIter=20
)

lr_model = lr.fit(train_ready)

predictions = lr_model.transform(test_ready)

print("Logistic Regression model trained successfully!")
print("Training records:", train_ready.count())
print("Testing records:", test_ready.count())

Logistic Regression model trained successfully!
Training records: 479028
Testing records: 213180


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

for metric in ["accuracy", "weightedPrecision", "weightedRecall"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="low_bikes_30min",
        predictionCol="prediction",
        metricName=metric
    )

    score = evaluator.evaluate(predictions)
    print(f"{metric}: {score:.4f}")

accuracy: 0.9575
weightedPrecision: 0.9578
weightedRecall: 0.9575


In [ ]:
predictions.groupBy(
    "low_bikes_30min",
    "prediction"
).count().orderBy(
    "low_bikes_30min",
    "prediction"
).show()

+---------------+----------+------+
|low_bikes_30min|prediction| count|
+---------------+----------+------+
|              0|       0.0|176287|
|              0|       1.0|  4851|
|              1|       0.0|  4219|
|              1|       1.0| 27823|
+---------------+----------+------+



In [ ]:
tp = 27823
fp = 4851
fn = 4219

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print(f"Low-bike precision: {precision:.4%}")
print(f"Low-bike recall: {recall:.4%}")
print(f"Low-bike F1 score: {f1:.4%}")

Low-bike precision: 85.1533%
Low-bike recall: 86.8329%
Low-bike F1 score: 85.9849%


In [ ]:
from datetime import date, timedelta

start_date = date(2026, 8, 22)

seven_day_paths = [
    s3_path + "nyc_date=" + (start_date + timedelta(days=i)).isoformat() + "/"
    for i in range(7)
]

for path in seven_day_paths:
    try:
        parquet_files = [
            f for f in dbutils.fs.ls(path)
            if f.name.endswith(".parquet")
        ]
        print(path.split("/")[-2], "-",
              len(parquet_files), "Parquet file(s)")
    except Exception as e:
        print("Unable to access:", path, str(e)[:100])

nyc_date=2026-08-22 - 1 Parquet file(s)
nyc_date=2026-08-23 - 1 Parquet file(s)
nyc_date=2026-08-24 - 1 Parquet file(s)
nyc_date=2026-08-25 - 1 Parquet file(s)
nyc_date=2026-08-26 - 1 Parquet file(s)
nyc_date=2026-08-27 - 1 Parquet file(s)
nyc_date=2026-08-28 - 1 Parquet file(s)


In [ ]:
seven_day_df = spark.read.parquet(*seven_day_paths)

print("Total records:", seven_day_df.count())

print(
    "Unique stations:",
    seven_day_df.select("station_id").distinct().count()
)

seven_day_df.select(
    F.min("fetched_at").alias("earliest_observation"),
    F.max("fetched_at").alias("latest_observation")
).show(truncate=False)

Total records: 5058923
Unique stations: 2510
+--------------------------+--------------------------+
|earliest_observation      |latest_observation        |
+--------------------------+--------------------------+
|2026-08-22 04:01:47.873904|2026-08-29 03:56:53.704357|
+--------------------------+--------------------------+



In [ ]:
seven_day_df.select(
    F.count("*").alias("total_records"),
    F.sum(F.col("station_id").isNull().cast("int")).alias("missing_station_ids"),
    F.sum(F.col("fetched_at").isNull().cast("int")).alias("missing_timestamps"),
    F.sum(F.col("num_bikes_available").isNull().cast("int")).alias("missing_bike_counts"),
    F.sum(
        (F.col("num_bikes_available") < 0).cast("int")
    ).alias("negative_bike_counts"),
    F.sum(
        F.col("_has_parse_error").cast("int")
    ).alias("parse_errors")
).show()

+-------------+-------------------+------------------+-------------------+--------------------+------------+
|total_records|missing_station_ids|missing_timestamps|missing_bike_counts|negative_bike_counts|parse_errors|
+-------------+-------------------+------------------+-------------------+--------------------+------------+
|      5058923|                  0|                 0|                  0|                   0|           0|
+-------------+-------------------+------------------+-------------------+--------------------+------------+



In [ ]:
duplicate_groups = (
    seven_day_df
    .groupBy("station_id", "fetched_at")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate station/timestamp combinations:",
    duplicate_groups.count()
)

Duplicate station/timestamp combinations: 0


In [ ]:
from pyspark.sql.window import Window

week_window = (
    Window.partitionBy("station_id")
    .orderBy("fetched_at")
)

week_intervals = (
    seven_day_df
    .withColumn(
        "previous_time",
        F.lag("fetched_at").over(week_window)
    )
    .withColumn(
        "minutes_between",
        (
            F.unix_timestamp("fetched_at")
            - F.unix_timestamp("previous_time")
        ) / 60
    )
)

week_intervals.select(
    F.round(F.min("minutes_between"), 2).alias("shortest"),
    F.round(F.avg("minutes_between"), 2).alias("average"),
    F.round(F.max("minutes_between"), 2).alias("longest"),
    F.sum(
        (F.col("minutes_between") > 10).cast("int")
    ).alias("gaps_over_10_minutes")
).show()

+--------+-------+-------+--------------------+
|shortest|average|longest|gaps_over_10_minutes|
+--------+-------+-------+--------------------+
|    1.52|    5.0|   6.95|                   0|
+--------+-------+-------+--------------------+



In [ ]:
week_model_df = (
    seven_day_df
    .withColumn(
        "bikes_30min_ahead",
        F.lead("num_bikes_available", 6).over(week_window)
    )
    .withColumn(
        "time_30min_ahead",
        F.lead("fetched_at", 6).over(week_window)
    )
    .withColumn(
        "minutes_ahead",
        (
            F.unix_timestamp("time_30min_ahead")
            - F.unix_timestamp("fetched_at")
        ) / 60
    )
    .withColumn(
        "low_bikes_30min",
        F.when(
            F.col("bikes_30min_ahead") < 3, 1
        ).when(
            F.col("bikes_30min_ahead").isNotNull(), 0
        )
    )
)

week_model_df.select(
    F.count("*").alias("total_records"),
    F.sum(F.col("low_bikes_30min").isNull().cast("int")).alias("missing_targets"),
    F.round(F.min("minutes_ahead"), 2).alias("shortest_prediction_window"),
    F.round(F.avg("minutes_ahead"), 2).alias("average_prediction_window"),
    F.round(F.max("minutes_ahead"), 2).alias("longest_prediction_window")
).show()

+-------------+---------------+--------------------------+-------------------------+-------------------------+
|total_records|missing_targets|shortest_prediction_window|average_prediction_window|longest_prediction_window|
+-------------+---------------+--------------------------+-------------------------+-------------------------+
|      5058923|          15060|                     24.93|                    29.99|                    31.93|
+-------------+---------------+--------------------------+-------------------------+-------------------------+



In [ ]:
week_model_df.select(
    F.sum(
        F.col("minutes_ahead").between(28, 32).cast("int")
    ).alias("valid_28_to_32_minutes"),

    F.sum(
        (F.col("minutes_ahead") < 28).cast("int")
    ).alias("too_short"),

    F.sum(
        (F.col("minutes_ahead") > 32).cast("int")
    ).alias("too_long")
).show()

+----------------------+---------+--------+
|valid_28_to_32_minutes|too_short|too_long|
+----------------------+---------+--------+
|               5028815|    15048|       0|
+----------------------+---------+--------+



In [ ]:
week_training_df = week_model_df.filter(
    F.col("minutes_ahead").between(28, 32)
    & F.col("low_bikes_30min").isNotNull()
)

week_training_df.groupBy("low_bikes_30min") \
    .count() \
    .orderBy("low_bikes_30min") \
    .show()

print(
    "Total usable observations:",
    week_training_df.count()
)

+---------------+-------+
|low_bikes_30min|  count|
+---------------+-------+
|              0|4268058|
|              1| 760757|
+---------------+-------+

Total usable observations: 5028815


In [ ]:
from datetime import datetime

split_time = datetime(2026, 8, 27, 4, 0, 0)

week_train_df = week_training_df.filter(
    F.col("time_30min_ahead") < F.lit(split_time)
)

week_test_df = week_training_df.filter(
    F.col("fetched_at") >= F.lit(split_time)
)

print("Training observations:", week_train_df.count())
print("Testing observations:", week_test_df.count())

print("\nTraining outcomes:")
week_train_df.groupBy("low_bikes_30min").count().show()

print("\nTesting outcomes:")
week_test_df.groupBy("low_bikes_30min").count().show()

Training observations: 3596747
Testing observations: 1417020

Training outcomes:
+---------------+-------+
|low_bikes_30min|  count|
+---------------+-------+
|              1| 555583|
|              0|3041164|
+---------------+-------+


Testing outcomes:
+---------------+-------+
|low_bikes_30min|  count|
+---------------+-------+
|              1| 202698|
|              0|1214322|
+---------------+-------+



In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

week_assembler = VectorAssembler(
    inputCols=["num_bikes_available"],
    outputCol="features"
)

week_train_ready = week_assembler.transform(week_train_df)
week_test_ready = week_assembler.transform(week_test_df)

week_lr = LogisticRegression(
    featuresCol="features",
    labelCol="low_bikes_30min",
    maxIter=20
)

week_lr_model = week_lr.fit(week_train_ready)

print("Seven-day logistic regression model trained successfully!")

Seven-day logistic regression model trained successfully!


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

week_predictions = week_lr_model.transform(week_test_ready)

week_accuracy = MulticlassClassificationEvaluator(
    labelCol="low_bikes_30min",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(week_predictions)

print(f"Seven-day model accuracy: {week_accuracy:.2%}")

print("\nConfusion Matrix:")
week_predictions.groupBy(
    "low_bikes_30min", "prediction"
).count().orderBy(
    "low_bikes_30min", "prediction"
).show()

Seven-day model accuracy: 97.33%

Confusion Matrix:
+---------------+----------+-------+
|low_bikes_30min|prediction|  count|
+---------------+----------+-------+
|              0|       0.0|1195160|
|              0|       1.0|  19162|
|              1|       0.0|  18645|
|              1|       1.0| 184053|
+---------------+----------+-------+



In [ ]:
tp = 184053
fp = 19162
fn = 18645

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print(f"Low-bike precision: {precision:.2%}")
print(f"Low-bike recall:    {recall:.2%}")
print(f"Low-bike F1-score:  {f1:.2%}")

Low-bike precision: 90.57%
Low-bike recall:    90.80%
Low-bike F1-score:  90.69%


In [ ]:
normal_count = 1214322
total_test = 1417020

baseline_accuracy = normal_count / total_test

print(f"Baseline accuracy: {baseline_accuracy:.2%}")
print(f"Our model accuracy: {week_accuracy:.2%}")
print(f"Improvement: {(week_accuracy - baseline_accuracy) * 100:.2f} percentage points")

Baseline accuracy: 85.70%
Our model accuracy: 97.33%
Improvement: 11.64 percentage points


## Team 5 — Citi Bike Seven-Day Model Evaluation

**Dataset:** August 22–28, 2026  
**Stations:** 2,510  
**Total observations:** 5,058,923  
**Usable labeled observations:** 5,028,815

### Model
- Algorithm: Logistic Regression
- Predictor: Current number of available bikes
- Target: Fewer than 3 bikes approximately 30 minutes later
- Training: August 22–26, 2026
- Testing: August 27–28, 2026
- Split: Chronological, with observations crossing the boundary excluded

### Results
- Baseline accuracy: 85.70%
- Model accuracy: 97.33%
- Improvement: 11.64 percentage points
- Low-bike precision: 90.57%
- Low-bike recall: 90.80%
- Low-bike F1-score: 90.69%

### Conclusion
The seven-day experiment demonstrated strong predictive performance
on the two-day holdout period. Additional testing across longer
periods and integration with the production pipeline are needed
before deployment.


In [ ]:
try:
    print("Model available:", week_lr_model is not None)
    print("Test data available:", week_test_ready is not None)
except NameError:
    print("Previous session variables need to be restored.")

Model available: True
Test data available: True


In [ ]:
week_predictions.select(
    "station_id",
    "fetched_at",
    "num_bikes_available",
    "bikes_30min_ahead",
    "low_bikes_30min",
    "prediction"
).show(10, truncate=False)

+------------------------------------+--------------------------+-------------------+-----------------+---------------+----------+
|station_id                          |fetched_at                |num_bikes_available|bikes_30min_ahead|low_bikes_30min|prediction|
+------------------------------------+--------------------------+-------------------+-----------------+---------------+----------+
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-27 04:01:47.86739 |0                  |0                |1              |1.0       |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-27 04:06:46.781405|0                  |0                |1              |1.0       |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-27 04:11:47.933401|0                  |0                |1              |1.0       |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-27 04:16:47.275819|0                  |1                |1              |1.0       |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-27 04:21:49.201615|0                 

In [ ]:
week_predictions.filter(
    (F.col("num_bikes_available") >= 3) &
    (F.col("prediction") == 1.0)
).select(
    "station_id",
    "fetched_at",
    "num_bikes_available",
    "bikes_30min_ahead",
    "low_bikes_30min",
    "prediction"
).show(10, truncate=False)

+----------+----------+-------------------+-----------------+---------------+----------+
|station_id|fetched_at|num_bikes_available|bikes_30min_ahead|low_bikes_30min|prediction|
+----------+----------+-------------------+-----------------+---------------+----------+
+----------+----------+-------------------+-----------------+---------------+----------+



In [ ]:
week_predictions.filter(
    (F.col("num_bikes_available") >= 3) &
    (F.col("low_bikes_30min") == 1)
).select(
    "station_id",
    "fetched_at",
    "num_bikes_available",
    "bikes_30min_ahead",
    "prediction"
).show(10, truncate=False)

+------------------------------------+--------------------------+-------------------+-----------------+----------+
|station_id                          |fetched_at                |num_bikes_available|bikes_30min_ahead|prediction|
+------------------------------------+--------------------------+-------------------+-----------------+----------+
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-28 11:51:49.246403|4                  |2                |0.0       |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-28 11:56:48.978999|4                  |2                |0.0       |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-28 12:02:12.124533|3                  |2                |0.0       |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-28 16:01:47.850659|3                  |2                |0.0       |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-28 16:06:47.440928|3                  |2                |0.0       |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-28 16:11:48.841459|3              

## 7. Model Evaluation and Emerging Shortage Detection

This section evaluates whether our machine learning models can identify stations that will experience a bike shortage approximately 30 minutes in advance.

Although the initial Logistic Regression model achieved 97.33% classification accuracy, it failed to identify emerging shortages at the default 50% prediction threshold.

The analysis investigates:

- Emerging shortage detection and recall
- Alternative prediction thresholds
- False-positive warnings
- Comparison with a simple bike-count warning rule
- The trade-off between detecting shortages and generating unnecessary alerts

The objective is to evaluate whether the models provide useful advance warnings, rather than simply recognizing stations that already have low bike availability.

In [ ]:
emerging_shortages = week_predictions.filter(
    (F.col("num_bikes_available") >= 3) &
    (F.col("low_bikes_30min") == 1)
)

total_emerging = emerging_shortages.count()

correct_emerging = emerging_shortages.filter(
    F.col("prediction") == 1.0
).count()

print("Total emerging shortages:", total_emerging)
print("Correctly predicted:", correct_emerging)

if total_emerging > 0:
    print(
        f"Emerging shortage recall: "
        f"{correct_emerging / total_emerging:.2%}"
    )

Total emerging shortages: 18645
Correctly predicted: 0
Emerging shortage recall: 0.00%


In [ ]:
trend_window = Window.partitionBy("station_id").orderBy("fetched_at")

trend_check_df = (
    seven_day_df
    .withColumn(
        "bikes_10min_ago",
        F.lag("num_bikes_available", 2).over(trend_window)
    )
    .withColumn(
        "bike_change_10min",
        F.col("num_bikes_available") - F.col("bikes_10min_ago")
    )
)

trend_check_df.select(
    "station_id",
    "fetched_at",
    "bikes_10min_ago",
    "num_bikes_available",
    "bike_change_10min"
).filter(
    F.col("bikes_10min_ago").isNotNull()
).show(10, truncate=False)

+------------------------------------+--------------------------+---------------+-------------------+-----------------+
|station_id                          |fetched_at                |bikes_10min_ago|num_bikes_available|bike_change_10min|
+------------------------------------+--------------------------+---------------+-------------------+-----------------+
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-22 04:11:47.930667|5              |5                  |0                |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-22 04:16:48.472727|5              |5                  |0                |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-22 04:21:47.052899|5              |5                  |0                |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-22 04:26:48.385417|5              |5                  |0                |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|2026-08-22 04:31:49.443111|5              |5                  |0                |
|0197e9da-ae03-40a7-8270-fe500bfc9e18|20

In [ ]:
trend_model_df = (
    week_training_df
    .join(
        trend_check_df.select(
            "station_id",
            "fetched_at",
            "bikes_10min_ago",
            "bike_change_10min"
        ),
        on=["station_id", "fetched_at"],
        how="inner"
    )
    .filter(
        F.col("bike_change_10min").isNotNull()
    )
)

print("Observations with trend information:", trend_model_df.count())

trend_model_df.select(
    "num_bikes_available",
    "bikes_10min_ago",
    "bike_change_10min",
    "low_bikes_30min"
).show(10)

Observations with trend information: 5023795
+-------------------+---------------+-----------------+---------------+
|num_bikes_available|bikes_10min_ago|bike_change_10min|low_bikes_30min|
+-------------------+---------------+-----------------+---------------+
|                  9|              9|                0|              0|
|                 10|             10|                0|              0|
|                  5|              5|                0|              0|
|                  7|              7|                0|              0|
|                  4|              4|                0|              0|
|                  1|              1|                0|              1|
|                 11|             11|                0|              0|
|                  7|              7|                0|              0|
|                  5|              6|               -1|              0|
|                 12|             12|                0|              0|
+------------------

In [ ]:
trend_train_df = trend_model_df.filter(
    F.col("time_30min_ahead") < F.lit(split_time)
)

trend_test_df = trend_model_df.filter(
    F.col("fetched_at") >= F.lit(split_time)
)

print("Training observations:", trend_train_df.count())
print("Testing observations:", trend_test_df.count())

print("\nTraining outcomes:")
trend_train_df.groupBy("low_bikes_30min").count().show()

print("\nTesting outcomes:")
trend_test_df.groupBy("low_bikes_30min").count().show()

Training observations: 3591727
Testing observations: 1417020

Training outcomes:
+---------------+-------+
|low_bikes_30min|  count|
+---------------+-------+
|              1| 554951|
|              0|3036776|
+---------------+-------+


Testing outcomes:
+---------------+-------+
|low_bikes_30min|  count|
+---------------+-------+
|              1| 202698|
|              0|1214322|
+---------------+-------+



In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

trend_assembler = VectorAssembler(
    inputCols=[
        "num_bikes_available",
        "bike_change_10min"
    ],
    outputCol="features"
)

trend_train_ready = trend_assembler.transform(trend_train_df)
trend_test_ready = trend_assembler.transform(trend_test_df)

trend_lr = LogisticRegression(
    featuresCol="features",
    labelCol="low_bikes_30min",
    maxIter=20
)

trend_lr_model = trend_lr.fit(trend_train_ready)

print("Two-feature logistic regression model trained successfully!")

Two-feature logistic regression model trained successfully!


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

trend_predictions = trend_lr_model.transform(trend_test_ready)

trend_accuracy = MulticlassClassificationEvaluator(
    labelCol="low_bikes_30min",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(trend_predictions)

print(f"Two-feature model accuracy: {trend_accuracy:.2%}")

print("\nConfusion Matrix:")
trend_predictions.groupBy(
    "low_bikes_30min", "prediction"
).count().orderBy(
    "low_bikes_30min", "prediction"
).show()

print("\nEmerging Shortage Detection:")
emerging_trend = trend_predictions.filter(
    (F.col("num_bikes_available") >= 3) &
    (F.col("low_bikes_30min") == 1)
)

total_emerging = emerging_trend.count()
correct_emerging = emerging_trend.filter(
    F.col("prediction") == 1.0
).count()

print("Total emerging shortages:", total_emerging)
print("Correctly predicted:", correct_emerging)

if total_emerging > 0:
    print(
        f"Emerging shortage recall: "
        f"{correct_emerging / total_emerging:.2%}"
    )

Two-feature model accuracy: 97.33%

Confusion Matrix:
+---------------+----------+-------+
|low_bikes_30min|prediction|  count|
+---------------+----------+-------+
|              0|       0.0|1195163|
|              0|       1.0|  19159|
|              1|       0.0|  18651|
|              1|       1.0| 184047|
+---------------+----------+-------+


Emerging Shortage Detection:
Total emerging shortages: 18645
Correctly predicted: 0
Emerging shortage recall: 0.00%


In [ ]:
trend_predictions.filter(
    (F.col("num_bikes_available") >= 3) &
    (F.col("low_bikes_30min") == 1)
).groupBy(
    "bike_change_10min"
).count().orderBy(
    F.desc("count")
).show(15)

+-----------------+-----+
|bike_change_10min|count|
+-----------------+-----+
|                0| 9431|
|               -1| 2873|
|                1| 2801|
|               -2| 1057|
|                2|  967|
|               -3|  442|
|                3|  394|
|               -4|  229|
|                4|  114|
|               -5|  111|
|               -6|   54|
|                5|   40|
|               -7|   26|
|               -8|   25|
|              -10|   12|
+-----------------+-----+
only showing top 15 rows


In [ ]:
trend_predictions.filter(
    (F.col("num_bikes_available") >= 3) &
    (F.col("low_bikes_30min") == 1)
).groupBy(
    "num_bikes_available"
).count().orderBy(
    "num_bikes_available"
).show(20)

+-------------------+-----+
|num_bikes_available|count|
+-------------------+-----+
|                  3|10372|
|                  4| 4006|
|                  5| 1663|
|                  6|  865|
|                  7|  552|
|                  8|  281|
|                  9|  196|
|                 10|  129|
|                 11|   74|
|                 12|   55|
|                 13|   36|
|                 14|   28|
|                 15|   32|
|                 16|   31|
|                 17|   31|
|                 18|   25|
|                 19|   31|
|                 20|   16|
|                 21|   15|
|                 22|   13|
+-------------------+-----+
only showing top 20 rows


In [ ]:
(
    trend_predictions
    .filter(F.col("num_bikes_available").between(3, 10))
    .groupBy("num_bikes_available")
    .agg(
        F.count("*").alias("total_observations"),
        F.sum("low_bikes_30min").alias("emerging_shortages")
    )
    .withColumn(
        "shortage_risk_percent",
        F.round(
            F.col("emerging_shortages") /
            F.col("total_observations") * 100, 2
        )
    )
    .orderBy("num_bikes_available")
    .show()
)

+-------------------+------------------+------------------+---------------------+
|num_bikes_available|total_observations|emerging_shortages|shortage_risk_percent|
+-------------------+------------------+------------------+---------------------+
|                  3|             48662|             10372|                21.31|
|                  4|             53468|              4006|                 7.49|
|                  5|             55310|              1663|                 3.01|
|                  6|             55324|               865|                 1.56|
|                  7|             54303|               552|                 1.02|
|                  8|             57349|               281|                 0.49|
|                  9|             54312|               196|                 0.36|
|                 10|             51757|               129|                 0.25|
+-------------------+------------------+------------------+---------------------+



In [ ]:
from pyspark.ml.functions import vector_to_array

probability_check = (
    trend_predictions
    .withColumn(
        "shortage_probability",
        vector_to_array("probability")[1]
    )
)

(
    probability_check
    .filter(F.col("num_bikes_available").between(3, 10))
    .groupBy("num_bikes_available")
    .agg(
        F.round(
            F.avg("shortage_probability") * 100, 2
        ).alias("average_predicted_risk_percent")
    )
    .orderBy("num_bikes_available")
    .show()
)

+-------------------+------------------------------+
|num_bikes_available|average_predicted_risk_percent|
+-------------------+------------------------------+
|                  3|                         36.53|
|                  4|                         15.23|
|                  5|                          5.31|
|                  6|                          1.72|
|                  7|                          0.54|
|                  8|                          0.17|
|                  9|                          0.05|
|                 10|                          0.02|
+-------------------+------------------------------+



In [ ]:
threshold_predictions = probability_check.withColumn(
    "prediction_20pct",
    F.when(F.col("shortage_probability") >= 0.20, 1).otherwise(0)
)

emerging_20pct = threshold_predictions.filter(
    (F.col("num_bikes_available") >= 3) &
    (F.col("low_bikes_30min") == 1)
)

total_emerging = emerging_20pct.count()

correct_emerging = emerging_20pct.filter(
    F.col("prediction_20pct") == 1
).count()

print("Total emerging shortages:", total_emerging)
print("Correctly predicted at 20% threshold:", correct_emerging)

print(
    f"Emerging shortage recall: "
    f"{correct_emerging / total_emerging:.2%}"
)

print("\nConfusion Matrix at 20% threshold:")
threshold_predictions.groupBy(
    "low_bikes_30min",
    "prediction_20pct"
).count().orderBy(
    "low_bikes_30min",
    "prediction_20pct"
).show()

Total emerging shortages: 18645
Correctly predicted at 20% threshold: 10372
Emerging shortage recall: 55.63%

Confusion Matrix at 20% threshold:
+---------------+----------------+-------+
|low_bikes_30min|prediction_20pct|  count|
+---------------+----------------+-------+
|              0|               0|1156878|
|              0|               1|  57444|
|              1|               0|   8273|
|              1|               1| 194425|
+---------------+----------------+-------+



In [ ]:
tp = 194425
fp = 57444
fn = 8273
tn = 1156878

precision_20 = tp / (tp + fp)
recall_20 = tp / (tp + fn)
f1_20 = 2 * precision_20 * recall_20 / (precision_20 + recall_20)
accuracy_20 = (tp + tn) / (tp + tn + fp + fn)

print(f"Accuracy at 20%:  {accuracy_20:.2%}")
print(f"Precision at 20%: {precision_20:.2%}")
print(f"Recall at 20%:    {recall_20:.2%}")
print(f"F1-score at 20%:  {f1_20:.2%}")

Accuracy at 20%:  95.36%
Precision at 20%: 77.19%
Recall at 20%:    95.92%
F1-score at 20%:  85.54%


In [ ]:
threshold_10 = probability_check.withColumn(
    "prediction_10pct",
    F.when(
        F.col("shortage_probability") >= 0.10, 1
    ).otherwise(0)
)

(
    threshold_10
    .filter(
        (F.col("num_bikes_available") >= 3) &
        (F.col("low_bikes_30min") == 1)
    )
    .groupBy("num_bikes_available")
    .agg(
        F.count("*").alias("actual_shortages"),
        F.sum("prediction_10pct").alias("detected_shortages")
    )
    .orderBy("num_bikes_available")
    .show(10)
)

+-------------------+----------------+------------------+
|num_bikes_available|actual_shortages|detected_shortages|
+-------------------+----------------+------------------+
|                  3|           10372|             10372|
|                  4|            4006|              4004|
|                  5|            1663|                 0|
|                  6|             865|                 0|
|                  7|             552|                 0|
|                  8|             281|                 0|
|                  9|             196|                 0|
|                 10|             129|                 0|
|                 11|              74|                 0|
|                 12|              55|                 0|
+-------------------+----------------+------------------+
only showing top 10 rows


In [ ]:
(
    threshold_10
    .groupBy("low_bikes_30min", "prediction_10pct")
    .count()
    .orderBy("low_bikes_30min", "prediction_10pct")
    .show()
)

+---------------+----------------+-------+
|low_bikes_30min|prediction_10pct|  count|
+---------------+----------------+-------+
|              0|               0|1107415|
|              0|               1| 106907|
|              1|               0|   4269|
|              1|               1| 198429|
+---------------+----------------+-------+



In [ ]:
(
    threshold_10
    .filter(
        (F.col("low_bikes_30min") == 0) &
        (F.col("prediction_10pct") == 1)
    )
    .groupBy("num_bikes_available")
    .count()
    .orderBy("num_bikes_available")
    .show(15)
)

+-------------------+-----+
|num_bikes_available|count|
+-------------------+-----+
|                  0| 2365|
|                  1| 4877|
|                  2|11919|
|                  3|38288|
|                  4|49458|
+-------------------+-----+



In [ ]:
simple_rule_df = probability_check.withColumn(
    "simple_warning",
    F.when(
        F.col("num_bikes_available") <= 4, 1
    ).otherwise(0)
)

print("Simple rule: Warn when 4 or fewer bikes are available")

simple_rule_df.groupBy(
    "low_bikes_30min",
    "simple_warning"
).count().orderBy(
    "low_bikes_30min",
    "simple_warning"
).show()

Simple rule: Warn when 4 or fewer bikes are available
+---------------+--------------+-------+
|low_bikes_30min|simple_warning|  count|
+---------------+--------------+-------+
|              0|             0|1107408|
|              0|             1| 106914|
|              1|             0|   4267|
|              1|             1| 198431|
+---------------+--------------+-------+



In [ ]:
hourly_risk = (
    probability_check
    .filter(F.col("num_bikes_available") >= 3)
    .withColumn(
        "hour_of_day",
        F.hour("fetched_at")
    )
    .groupBy("hour_of_day")
    .agg(
        F.count("*").alias("total_observations"),
        F.sum("low_bikes_30min").alias("emerging_shortages")
    )
    .withColumn(
        "shortage_risk_percent",
        F.round(
            F.col("emerging_shortages") /
            F.col("total_observations") * 100,
            2
        )
    )
    .orderBy("hour_of_day")
)

hourly_risk.show(24)

+-----------+------------------+------------------+---------------------+
|hour_of_day|total_observations|emerging_shortages|shortage_risk_percent|
+-----------+------------------+------------------+---------------------+
|          0|             51702|               851|                 1.65|
|          1|             52143|               752|                 1.44|
|          2|             52396|               785|                  1.5|
|          3|             39419|               461|                 1.17|
|          4|             51587|               337|                 0.65|
|          5|             51724|               270|                 0.52|
|          6|             51824|               181|                 0.35|
|          7|             51914|               161|                 0.31|
|          8|             51883|               188|                 0.36|
|          9|             52092|               280|                 0.54|
|         10|             52744|      

In [ ]:
print("Spark session timezone:")
print(spark.conf.get("spark.sql.session.timeZone"))

print("\nSample timestamps:")
seven_day_df.select(
    "fetched_at"
).orderBy("fetched_at").show(5, truncate=False)

Spark session timezone:
Etc/UTC

Sample timestamps:
+--------------------------+
|fetched_at                |
+--------------------------+
|2026-08-22 04:01:47.873904|
|2026-08-22 04:01:47.873904|
|2026-08-22 04:01:47.873904|
|2026-08-22 04:01:47.873904|
|2026-08-22 04:01:47.873904|
+--------------------------+
only showing top 5 rows


In [ ]:
from pyspark.sql import functions as F

time_feature_df = probability_check.withColumn(
    "nyc_timestamp",
    F.from_utc_timestamp(
        F.col("fetched_at"),
        "America/New_York"
    )
).withColumn(
    "nyc_hour",
    F.hour("nyc_timestamp")
)

time_feature_df.select(
    "fetched_at",
    "nyc_timestamp",
    "nyc_hour"
).orderBy("fetched_at").show(10, truncate=False)

+-------------------------+-------------------------+--------+
|fetched_at               |nyc_timestamp            |nyc_hour|
+-------------------------+-------------------------+--------+
|2026-08-27 04:01:47.86739|2026-08-27 00:01:47.86739|0       |
|2026-08-27 04:01:47.86739|2026-08-27 00:01:47.86739|0       |
|2026-08-27 04:01:47.86739|2026-08-27 00:01:47.86739|0       |
|2026-08-27 04:01:47.86739|2026-08-27 00:01:47.86739|0       |
|2026-08-27 04:01:47.86739|2026-08-27 00:01:47.86739|0       |
|2026-08-27 04:01:47.86739|2026-08-27 00:01:47.86739|0       |
|2026-08-27 04:01:47.86739|2026-08-27 00:01:47.86739|0       |
|2026-08-27 04:01:47.86739|2026-08-27 00:01:47.86739|0       |
|2026-08-27 04:01:47.86739|2026-08-27 00:01:47.86739|0       |
|2026-08-27 04:01:47.86739|2026-08-27 00:01:47.86739|0       |
+-------------------------+-------------------------+--------+
only showing top 10 rows


In [ ]:
(
    time_feature_df
    .filter(F.col("num_bikes_available") == 4)
    .groupBy("nyc_hour")
    .agg(
        F.count("*").alias("total_observations"),
        F.sum("low_bikes_30min").alias("shortages_30min")
    )
    .withColumn(
        "shortage_risk_percent",
        F.round(
            F.col("shortages_30min") /
            F.col("total_observations") * 100,
            2
        )
    )
    .orderBy("nyc_hour")
    .show(24)
)

+--------+------------------+---------------+---------------------+
|nyc_hour|total_observations|shortages_30min|shortage_risk_percent|
+--------+------------------+---------------+---------------------+
|       0|              1935|             68|                 3.51|
|       1|              1962|             55|                  2.8|
|       2|              1958|              7|                 0.36|
|       3|              1968|             16|                 0.81|
|       4|              1932|             11|                 0.57|
|       5|              1922|             30|                 1.56|
|       6|              2239|             85|                  3.8|
|       7|              1981|            209|                10.55|
|       8|              2014|            287|                14.25|
|       9|              2322|            284|                12.23|
|      10|              2549|            232|                  9.1|
|      11|              2383|            265|   

In [ ]:
def add_nyc_hour(df):
    return (
        df.withColumn(
            "nyc_hour",
            F.hour(
                F.from_utc_timestamp(
                    F.col("fetched_at"),
                    "America/New_York"
                )
            )
        )
    )

time_train_df = add_nyc_hour(trend_train_df)
time_test_df = add_nyc_hour(trend_test_df)

print("Training observations:", time_train_df.count())
print("Testing observations:", time_test_df.count())

print("\nSample time features:")
time_test_df.select(
    "fetched_at",
    "nyc_hour",
    "num_bikes_available",
    "bike_change_10min",
    "low_bikes_30min"
).show(5, truncate=False)

Training observations: 3591727
Testing observations: 1417020

Sample time features:
+--------------------------+--------+-------------------+-----------------+---------------+
|fetched_at                |nyc_hour|num_bikes_available|bike_change_10min|low_bikes_30min|
+--------------------------+--------+-------------------+-----------------+---------------+
|2026-08-27 04:01:47.86739 |0       |11                 |0                |0              |
|2026-08-27 04:06:46.781405|0       |11                 |0                |0              |
|2026-08-27 04:11:47.933401|0       |11                 |0                |0              |
|2026-08-27 04:16:47.275819|0       |11                 |0                |0              |
|2026-08-27 04:21:49.201615|0       |11                 |0                |0              |
+--------------------------+--------+-------------------+-----------------+---------------+
only showing top 5 rows


In [ ]:
import math

def add_cyclical_hour(df):
    return (
        df.withColumn(
            "hour_sin",
            F.sin(2 * math.pi * F.col("nyc_hour") / 24)
        )
        .withColumn(
            "hour_cos",
            F.cos(2 * math.pi * F.col("nyc_hour") / 24)
        )
    )

cyclical_train_df = add_cyclical_hour(time_train_df)
cyclical_test_df = add_cyclical_hour(time_test_df)

cyclical_test_df.select(
    "nyc_hour",
    "hour_sin",
    "hour_cos"
).distinct().orderBy("nyc_hour").show(24)

+--------+--------------------+--------------------+
|nyc_hour|            hour_sin|            hour_cos|
+--------+--------------------+--------------------+
|       0|                 0.0|                 1.0|
|       1| 0.25881904510252074|  0.9659258262890683|
|       2| 0.49999999999999994|  0.8660254037844387|
|       3|  0.7071067811865475|  0.7071067811865476|
|       4|  0.8660254037844386|  0.5000000000000001|
|       5|  0.9659258262890683| 0.25881904510252074|
|       6|                 1.0|6.123233995736766...|
|       7|  0.9659258262890683|-0.25881904510252063|
|       8|  0.8660254037844387| -0.4999999999999998|
|       9|  0.7071067811865476| -0.7071067811865475|
|      10| 0.49999999999999994| -0.8660254037844387|
|      11|   0.258819045102521| -0.9659258262890682|
|      12|1.224646799147353...|                -1.0|
|      13| -0.2588190451025208| -0.9659258262890683|
|      14| -0.4999999999999997| -0.8660254037844388|
|      15| -0.7071067811865471| -0.70710678118

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Combine our four features
time_assembler = VectorAssembler(
    inputCols=[
        "num_bikes_available",
        "bike_change_10min",
        "hour_sin",
        "hour_cos"
    ],
    outputCol="features"
)

# Prepare training and testing data
time_train_ready = time_assembler.transform(cyclical_train_df)
time_test_ready = time_assembler.transform(cyclical_test_df)

# Train our third logistic regression model
time_lr = LogisticRegression(
    featuresCol="features",
    labelCol="low_bikes_30min",
    maxIter=20
)

time_lr_model = time_lr.fit(time_train_ready)

# Generate predictions on our test data
time_predictions = time_lr_model.transform(time_test_ready)

print("Time-aware logistic regression model trained successfully!")

print("\nConfusion Matrix:")
time_predictions.groupBy(
    "low_bikes_30min",
    "prediction"
).count().orderBy(
    "low_bikes_30min",
    "prediction"
).show()

Time-aware logistic regression model trained successfully!

Confusion Matrix:
+---------------+----------+-------+
|low_bikes_30min|prediction|  count|
+---------------+----------+-------+
|              0|       0.0|1195165|
|              0|       1.0|  19157|
|              1|       0.0|  18652|
|              1|       1.0| 184046|
+---------------+----------+-------+



In [ ]:
from pyspark.ml.functions import vector_to_array

time_10_predictions = (
    time_predictions
    .withColumn(
        "shortage_probability",
        vector_to_array("probability")[1]
    )
    .withColumn(
        "prediction_10pct",
        F.when(
            F.col("shortage_probability") >= 0.10, 1
        ).otherwise(0)
    )
)

emerging_time_10 = time_10_predictions.filter(
    (F.col("num_bikes_available") >= 3) &
    (F.col("low_bikes_30min") == 1)
)

total_emerging = emerging_time_10.count()

detected_emerging = emerging_time_10.filter(
    F.col("prediction_10pct") == 1
).count()

print("Total emerging shortages:", total_emerging)
print("Detected emerging shortages:", detected_emerging)

print(
    f"Emerging-shortage recall: "
    f"{detected_emerging / total_emerging:.2%}"
)

print("\nConfusion Matrix at 10% threshold:")

time_10_predictions.groupBy(
    "low_bikes_30min",
    "prediction_10pct"
).count().orderBy(
    "low_bikes_30min",
    "prediction_10pct"
).show()

Total emerging shortages: 18645
Detected emerging shortages: 14376
Emerging-shortage recall: 77.10%

Confusion Matrix at 10% threshold:
+---------------+----------------+-------+
|low_bikes_30min|prediction_10pct|  count|
+---------------+----------------+-------+
|              0|               0|1107414|
|              0|               1| 106908|
|              1|               0|   4269|
|              1|               1| 198429|
+---------------+----------------+-------+



### 7.1 Random Forest Model and Feature Importance

This section evaluates a Random Forest classifier using four features:

- Current number of available bikes
- Change in bike availability over approximately 10 minutes
- Time of day represented by sine
- Time of day represented by cosine

The analysis compares Random Forest predictions with observed shortage outcomes and examines feature importance and predicted shortage probabilities.

The objective is to determine whether Random Forest captures useful patterns beyond those identified by Logistic Regression and the simple bike-count warning rule.

In [ ]:
from pyspark.ml.feature import VectorAssembler

rf_assembler = VectorAssembler(
    inputCols=[
        "num_bikes_available",
        "bike_change_10min",
        "hour_sin",
        "hour_cos"
    ],
    outputCol="rf_features"
)

rf_train_ready = rf_assembler.transform(
    cyclical_train_df
)

rf_test_ready = rf_assembler.transform(
    cyclical_test_df
)

print("Random Forest training observations:",
      rf_train_ready.count())

print("Random Forest testing observations:",
      rf_test_ready.count())

rf_train_ready.select(
    "num_bikes_available",
    "bike_change_10min",
    "hour_sin",
    "hour_cos",
    "rf_features",
    "low_bikes_30min"
).show(5, truncate=False)

Random Forest training observations: 3591727
Random Forest testing observations: 1417020
+-------------------+-----------------+--------+--------+-----------------+---------------+
|num_bikes_available|bike_change_10min|hour_sin|hour_cos|rf_features      |low_bikes_30min|
+-------------------+-----------------+--------+--------+-----------------+---------------+
|8                  |1                |0.0     |1.0     |[8.0,1.0,0.0,1.0]|0              |
|8                  |1                |0.0     |1.0     |[8.0,1.0,0.0,1.0]|0              |
|9                  |1                |0.0     |1.0     |[9.0,1.0,0.0,1.0]|0              |
|9                  |1                |0.0     |1.0     |[9.0,1.0,0.0,1.0]|0              |
|9                  |0                |0.0     |1.0     |[9.0,0.0,0.0,1.0]|0              |
+-------------------+-----------------+--------+--------+-----------------+---------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# Create our Random Forest model
rf_model_setup = RandomForestClassifier(
    featuresCol="rf_features",
    labelCol="low_bikes_30min",
    numTrees=20,
    maxDepth=5,
    seed=42
)

# Train the model
rf_model = rf_model_setup.fit(rf_train_ready)

print("Random Forest model trained successfully!")

# Generate predictions
rf_predictions = rf_model.transform(rf_test_ready)

print("\nRandom Forest Confusion Matrix:")

rf_predictions.groupBy(
    "low_bikes_30min",
    "prediction"
).count().orderBy(
    "low_bikes_30min",
    "prediction"
).show()

Random Forest model trained successfully!

Random Forest Confusion Matrix:
+---------------+----------+-------+
|low_bikes_30min|prediction|  count|
+---------------+----------+-------+
|              0|       0.0|1195160|
|              0|       1.0|  19162|
|              1|       0.0|  18645|
|              1|       1.0| 184053|
+---------------+----------+-------+



In [ ]:
feature_names = [
    "num_bikes_available",
    "bike_change_10min",
    "hour_sin",
    "hour_cos"
]

importance_scores = rf_model.featureImportances.toArray()

print("Random Forest Feature Importance")
print("-" * 45)

for feature, importance in sorted(
    zip(feature_names, importance_scores),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{feature}: {importance:.4f}")

Random Forest Feature Importance
---------------------------------------------
num_bikes_available: 0.9903
bike_change_10min: 0.0082
hour_cos: 0.0009
hour_sin: 0.0006


In [ ]:
from pyspark.ml.functions import vector_to_array

rf_probability_df = rf_predictions.withColumn(
    "shortage_probability",
    vector_to_array("probability")[1]
)

(
    rf_probability_df
    .filter(F.col("num_bikes_available").between(3, 10))
    .groupBy("num_bikes_available")
    .agg(
        F.round(
            F.avg("shortage_probability") * 100,
            2
        ).alias("average_predicted_risk_percent"),
        F.round(
            F.avg("low_bikes_30min") * 100,
            2
        ).alias("actual_shortage_rate_percent")
    )
    .orderBy("num_bikes_available")
    .show()
)

+-------------------+------------------------------+----------------------------+
|num_bikes_available|average_predicted_risk_percent|actual_shortage_rate_percent|
+-------------------+------------------------------+----------------------------+
|                  3|                          5.09|                       21.31|
|                  4|                           2.3|                        7.49|
|                  5|                          2.25|                        3.01|
|                  6|                          2.25|                        1.56|
|                  7|                          2.25|                        1.02|
|                  8|                          2.26|                        0.49|
|                  9|                          2.24|                        0.36|
|                 10|                          2.25|                        0.25|
+-------------------+------------------------------+----------------------------+



In [ ]:
(
    time_test_df
    .filter(F.col("num_bikes_available") == 4)
    .groupBy("station_id")
    .agg(
        F.count("*").alias("observations"),
        F.sum("low_bikes_30min").alias("shortages")
    )
    .filter(F.col("observations") >= 10)
    .withColumn(
        "shortage_rate_percent",
        F.round(
            F.col("shortages") /
            F.col("observations") * 100,
            2
        )
    )
    .orderBy(F.desc("shortage_rate_percent"))
    .show(15, truncate=False)
)

+------------------------------------+------------+---------+---------------------+
|station_id                          |observations|shortages|shortage_rate_percent|
+------------------------------------+------------+---------+---------------------+
|ed279ef2-52ee-4ff2-9a52-edd8137ea034|10          |8        |80.0                 |
|66db33fc-0aca-11e7-82f6-3863bb44ef7c|15          |11       |73.33                |
|1963272613486484230                 |10          |6        |60.0                 |
|66dc292c-0aca-11e7-82f6-3863bb44ef7c|15          |9        |60.0                 |
|b442a648-e9f4-4893-951a-64d258bc0e55|12          |7        |58.33                |
|b1cdec34-d4f9-4313-be82-729bb9f7bb18|19          |11       |57.89                |
|1880644037244334512                 |23          |13       |56.52                |
|72196024-6f2b-45cb-9764-609911c7dc0b|18          |10       |55.56                |
|9c1c3499-1b11-471f-b59a-87d9e27e15c5|11          |6        |54.55          

### 7.2 Station-Level Shortage Patterns

This section investigates whether individual Citi Bike stations exhibit recurring shortage patterns.

Using observations where exactly four bikes are available, we compare each station's shortage rate during the training period with its shortage rate during the later testing period.

Across 830 stations meeting the initial observation requirements, the Pearson correlation between training and testing shortage rates was 0.4923. The correlation remained moderately positive when higher minimum observation counts were required.

These findings suggest that historical station behavior may provide useful predictive information beyond current bike availability.

However, the analysis is exploratory. The dataset covers only one week, and repeated observations may represent overlapping shortage events. Further validation is needed before incorporating station-level historical risk into a production model.

In [ ]:
station_train_risk = (
    trend_train_df
    .filter(F.col("num_bikes_available") == 4)
    .groupBy("station_id")
    .agg(
        F.count("*").alias("train_observations"),
        F.sum("low_bikes_30min").alias("train_shortages")
    )
    .withColumn(
        "train_shortage_rate",
        F.round(
            F.col("train_shortages") /
            F.col("train_observations") * 100,
            2
        )
    )
)

print("Stations with at least 30 training observations:")

(
    station_train_risk
    .filter(F.col("train_observations") >= 30)
    .orderBy(F.desc("train_shortage_rate"))
    .show(15, truncate=False)
)

Stations with at least 30 training observations:
+------------------------------------+------------------+---------------+-------------------+
|station_id                          |train_observations|train_shortages|train_shortage_rate|
+------------------------------------+------------------+---------------+-------------------+
|956e9f8d-d23a-4a0e-bf28-84180e818af7|43                |25             |58.14              |
|490ca6cc-c2a2-4838-acf0-4704ed60dc51|33                |19             |57.58              |
|66dd515d-0aca-11e7-82f6-3863bb44ef7c|33                |18             |54.55              |
|904930bd-2671-4695-9c1a-8a894b041129|32                |17             |53.13              |
|29b63342-02ab-4a62-be64-d3e59c0e1988|59                |30             |50.85              |
|66dc79e6-0aca-11e7-82f6-3863bb44ef7c|44                |22             |50.0               |
|66dcbf69-0aca-11e7-82f6-3863bb44ef7c|70                |35             |50.0               |
|66dcc44a-0

In [ ]:
station_test_risk = (
    trend_test_df
    .filter(F.col("num_bikes_available") == 4)
    .groupBy("station_id")
    .agg(
        F.count("*").alias("test_observations"),
        F.sum("low_bikes_30min").alias("test_shortages")
    )
    .withColumn(
        "test_shortage_rate",
        F.round(
            F.col("test_shortages") /
            F.col("test_observations") * 100,
            2
        )
    )
)

station_comparison = (
    station_train_risk
    .join(station_test_risk, on="station_id", how="inner")
    .filter(
        (F.col("train_observations") >= 30) &
        (F.col("test_observations") >= 10)
    )
)

print("Stations with sufficient observations in both periods:")

station_comparison.select(
    "station_id",
    "train_observations",
    "train_shortage_rate",
    "test_observations",
    "test_shortage_rate"
).orderBy(
    F.desc("train_shortage_rate")
).show(15, truncate=False)

Stations with sufficient observations in both periods:
+------------------------------------+------------------+-------------------+-----------------+------------------+
|station_id                          |train_observations|train_shortage_rate|test_observations|test_shortage_rate|
+------------------------------------+------------------+-------------------+-----------------+------------------+
|956e9f8d-d23a-4a0e-bf28-84180e818af7|43                |58.14              |25               |52.0              |
|490ca6cc-c2a2-4838-acf0-4704ed60dc51|33                |57.58              |29               |24.14             |
|66dd515d-0aca-11e7-82f6-3863bb44ef7c|33                |54.55              |14               |42.86             |
|66dcbf69-0aca-11e7-82f6-3863bb44ef7c|70                |50.0               |39               |30.77             |
|66dcc44a-0aca-11e7-82f6-3863bb44ef7c|51                |49.02              |15               |20.0              |
|c2bb1874-bcb7-47c5-aa7d-

In [ ]:
print(
    "Stations included:",
    station_comparison.count()
)

correlation = station_comparison.stat.corr(
    "train_shortage_rate",
    "test_shortage_rate"
)

print(
    f"Correlation between training and testing shortage rates: "
    f"{correlation:.4f}"
)

Stations included: 830
Correlation between training and testing shortage rates: 0.4923


In [ ]:
for minimum in [10, 20, 30, 50]:
    filtered = station_comparison.filter(
        (F.col("train_observations") >= minimum) &
        (F.col("test_observations") >= minimum)
    )

    station_count = filtered.count()

    correlation = (
        filtered.stat.corr(
            "train_shortage_rate",
            "test_shortage_rate"
        )
        if station_count >= 2 else None
    )

    print(
        f"Minimum {minimum} observations in each period | "
        f"Stations: {station_count} | "
        f"Correlation: {correlation}"
    )

Minimum 10 observations in each period | Stations: 830 | Correlation: 0.49231620980471413
Minimum 20 observations in each period | Stations: 648 | Correlation: 0.5427524535898101
Minimum 30 observations in each period | Stations: 511 | Correlation: 0.5226735016339794
Minimum 50 observations in each period | Stations: 292 | Correlation: 0.5765436527137039


## 8. Conclusions and Future Improvements

### Key Findings

This project examined whether historical Citi Bike station-status data could predict bike shortages approximately 30 minutes in advance.

The initial Logistic Regression and Random Forest models achieved approximately 97.33% classification accuracy. However, at the default 50% classification threshold, neither model successfully identified emerging shortages at stations that currently had at least three bikes available.

Lowering the Logistic Regression prediction threshold to 10% increased emerging-shortage detection to approximately 77.10%, but also produced more false-positive warnings. A simple rule that warned when four or fewer bikes were available performed almost identically.

Random Forest feature importance indicated that current bike availability dominated the model's decisions, while recent bike changes and time-of-day features contributed relatively little.

Exploratory station-level analysis revealed a moderate positive correlation between historical and later shortage rates, suggesting that station-specific behavior may be useful for future modeling.

### Limitations

- The dataset covers approximately one week of historical observations.
- Consecutive observations may represent overlapping shortage events.
- The testing period was used repeatedly during exploratory analysis, so reported performance should not be treated as an unbiased final estimate.
- Weather and station-specific historical risk were not incorporated into the trained models.
- Production deployment and ongoing model monitoring have not yet been demonstrated in this notebook.

### Recommendations

Future development should prioritize station-specific historical demand patterns, additional weather and calendar features, and evaluation on a new chronological holdout dataset.

Model selection should emphasize advance-warning performance and false-alarm rates rather than overall accuracy alone.

### Conclusion

The analysis demonstrates that high classification accuracy does not necessarily translate into useful operational predictions. Although the initial models did not substantially outperform a simple bike-count warning rule, the experiments identified promising directions for improving future Citi Bike shortage predictions.